# Trading Data → DuckDB Integration

这个 notebook 做 4 件事：
1. 扫描 `trading_data/*.csv` 的字段（attribute）与 schema 差异
2. 导出字段统计报告
3. 将全部文件写入 `trading_data.duckdb`
4. 提供基础校验查询

In [4]:
from pathlib import Path
from collections import Counter
import pandas as pd
import duckdb

ROOT = Path('.')
DATA_DIR = ROOT / 'trading_data'
DB_PATH = ROOT / 'trading_data.duckdb'
ATTR_REPORT = ROOT / 'trading_data_attribute_frequency.csv'
SCHEMA_REPORT = ROOT / 'trading_data_schema_frequency.csv'

ENCODINGS = ['utf-8-sig', 'gb18030', 'gbk']
files = sorted(DATA_DIR.glob('*.csv'))
print(f'CSV file count: {len(files)}')

CSV file count: 5739


In [5]:
def read_header_with_encoding(path: Path):
    last_err = None
    for enc in ENCODINGS:
        try:
            cols = list(pd.read_csv(path, nrows=0, encoding=enc).columns)
            cols = [str(c).strip() for c in cols]
            return cols, enc
        except Exception as e:
            last_err = e
    raise RuntimeError(f'Cannot read header: {path.name} | {last_err}')

header_map = {}
encoding_map = {}
attribute_counter = Counter()
schema_counter = Counter()

for i, f in enumerate(files, 1):
    cols, enc = read_header_with_encoding(f)
    header_map[f] = cols
    encoding_map[f] = enc
    attribute_counter.update(cols)
    schema_counter[tuple(cols)] += 1
    if i % 500 == 0:
        print(f'Header scanned: {i}/{len(files)}')

all_attributes = sorted(attribute_counter.keys())
print('Unique attributes:', len(all_attributes))
print('Unique schemas:', len(schema_counter))

Header scanned: 500/5739
Header scanned: 1000/5739
Header scanned: 1500/5739
Header scanned: 2000/5739
Header scanned: 2500/5739
Header scanned: 3000/5739
Header scanned: 3500/5739
Header scanned: 4000/5739
Header scanned: 4500/5739
Header scanned: 5000/5739
Header scanned: 5500/5739
Unique attributes: 41
Unique schemas: 5


In [3]:
attr_df = pd.DataFrame({
    'attribute': list(attribute_counter.keys()),
    'file_count': list(attribute_counter.values())
}).sort_values(['file_count', 'attribute'], ascending=[False, True]).reset_index(drop=True)

schema_rows = []
for schema, cnt in schema_counter.items():
    schema_rows.append({
        'file_count': cnt,
        'column_count': len(schema),
        'columns_joined': ' | '.join(schema)
    })
schema_df = pd.DataFrame(schema_rows).sort_values('file_count', ascending=False).reset_index(drop=True)

attr_df.to_csv(ATTR_REPORT, index=False, encoding='utf-8-sig')
schema_df.to_csv(SCHEMA_REPORT, index=False, encoding='utf-8-sig')

display(attr_df.head(30))
display(schema_df.head(10))
print(f'Saved: {ATTR_REPORT}')
print(f'Saved: {SCHEMA_REPORT}')

,attribute,file_count
0,交易日期,5739
1,前收盘价,5739
2,开盘价,5739
3,成交量(手),5739
4,收盘价,5739
5,最低价,5739
6,最高价,5739
7,涨跌幅(%),5739
8,涨跌额,5739
9,股票代码,5739


,file_count,column_count,columns_joined
0,5124,34,股票代码 | 名称 | 所属行业 | 地域 | 上市日期 | TS代码 | 交易日期 | 开...
1,607,33,股票代码 | 名称 | 所属行业 | 地域 | 上市日期 | TS代码 | 交易日期 | 开...
2,6,21,股票代码 | 交易日期 | 开盘价 | 最高价 | 最低价 | 收盘价 | 前收盘价 | 涨...
3,1,31,股票代码 | 名称 | 所属行业 | 地域 | 上市日期 | TS代码 | 交易日期 | 开...
4,1,18,股票代码 | 名称 | 所属行业 | 地域 | 上市日期 | TS代码 | 交易日期 | 开...


Saved: trading_data_attribute_frequency.csv
Saved: trading_data_schema_frequency.csv


In [4]:
if DB_PATH.exists():
    DB_PATH.unlink()

db_columns = all_attributes + ['source_file']

def q(col: str) -> str:
    return '"' + col.replace('"', '""') + '"'

create_cols_sql = ', '.join([f"{q(c)} VARCHAR" for c in db_columns])
create_sql = f"CREATE TABLE trading_data_raw ({create_cols_sql});"

conn = duckdb.connect(str(DB_PATH))
conn.execute(create_sql)
print(f'Created DB: {DB_PATH}')
print(f'Raw column count: {len(db_columns)}')

Created DB: trading_data.duckdb
Raw column count: 42


In [5]:
inserted_rows = 0
for i, f in enumerate(files, 1):
    enc = encoding_map[f]
    df = pd.read_csv(f, encoding=enc)
    df.columns = [str(c).strip() for c in df.columns]

    missing_cols = [c for c in all_attributes if c not in df.columns]
    for c in missing_cols:
        df[c] = None

    ordered = df[all_attributes].copy()
    ordered['source_file'] = f.name

    conn.register('tmp_df', ordered)
    conn.execute('INSERT INTO trading_data_raw SELECT * FROM tmp_df')
    conn.unregister('tmp_df')

    inserted_rows += len(ordered)
    if i % 200 == 0 or i == len(files):
        print(f'Loaded files: {i}/{len(files)} | rows inserted: {inserted_rows:,}')

print(f'Total inserted rows: {inserted_rows:,}')

Loaded files: 200/5739 | rows inserted: 1,172,831
Loaded files: 400/5739 | rows inserted: 2,359,886
Loaded files: 600/5739 | rows inserted: 2,917,416
Loaded files: 800/5739 | rows inserted: 3,763,051
Loaded files: 1000/5739 | rows inserted: 4,497,919
Loaded files: 1200/5739 | rows inserted: 5,157,544
Loaded files: 1400/5739 | rows inserted: 5,598,012
Loaded files: 1600/5739 | rows inserted: 6,114,883
Loaded files: 1800/5739 | rows inserted: 6,789,289
Loaded files: 2000/5739 | rows inserted: 7,304,576
Loaded files: 2200/5739 | rows inserted: 7,714,510
Loaded files: 2400/5739 | rows inserted: 7,981,047
Loaded files: 2600/5739 | rows inserted: 8,180,967
Loaded files: 2800/5739 | rows inserted: 8,326,800
Loaded files: 3000/5739 | rows inserted: 8,931,448
Loaded files: 3200/5739 | rows inserted: 10,107,719
Loaded files: 3400/5739 | rows inserted: 11,217,660
Loaded files: 3600/5739 | rows inserted: 12,419,451
Loaded files: 3800/5739 | rows inserted: 13,056,093
Loaded files: 4000/5739 | rows 

In [6]:
try:
    conn.close()
except Exception:
    pass
conn = duckdb.connect(str(DB_PATH))

def first_existing(candidates):
    for c in candidates:
        if c in all_attributes:
            return c
    return None


def text_expr(candidates, alias):
    col = first_existing(candidates)
    if col is None:
        return f"NULL::VARCHAR AS {alias}"
    return f'TRIM("{col}")::VARCHAR AS {alias}'


def date_expr(candidates, alias):
    col = first_existing(candidates)
    if col is None:
        return f"NULL::DATE AS {alias}"
    return f"""
    CASE
        WHEN length(trim(\"{col}\")) = 8 THEN strptime(trim(\"{col}\"), '%Y%m%d')::DATE
        WHEN trim(\"{col}\") LIKE '____-__-__' THEN strptime(trim(\"{col}\"), '%Y-%m-%d')::DATE
        ELSE NULL
    END AS {alias}
    """.strip()


def numeric_expr(candidates, alias, multiplier=1.0):
    col = first_existing(candidates)
    if col is None:
        return f"NULL::DOUBLE AS {alias}"
    if multiplier == 1.0:
        return f'TRY_CAST("{col}" AS DOUBLE) AS {alias}'
    return f'(TRY_CAST("{col}" AS DOUBLE) * {multiplier}) AS {alias}'


turnover_amount_expr = numeric_expr(['成交金额(元)', '成交额(元)', '成交额(千元)', '成交额(万元)'], 'turnover_amount')
if first_existing(['成交额(千元)']) is not None and first_existing(['成交金额(元)', '成交额(元)']) is None:
    turnover_amount_expr = numeric_expr(['成交额(千元)'], 'turnover_amount', 1000.0)
elif first_existing(['成交额(万元)']) is not None and first_existing(['成交金额(元)', '成交额(元)', '成交额(千元)']) is None:
    turnover_amount_expr = numeric_expr(['成交额(万元)'], 'turnover_amount', 10000.0)

free_float_expr = numeric_expr(['流通市值(元)', '流通市值(万元)'], 'free_float_mkt_cap')
if first_existing(['流通市值(万元)']) is not None and first_existing(['流通市值(元)']) is None:
    free_float_expr = numeric_expr(['流通市值(万元)'], 'free_float_mkt_cap', 10000.0)

total_mkt_cap_expr = numeric_expr(['总市值(元)', '总市值(万元)'], 'total_mkt_cap')
if first_existing(['总市值(万元)']) is not None and first_existing(['总市值(元)']) is None:
    total_mkt_cap_expr = numeric_expr(['总市值(万元)'], 'total_mkt_cap', 10000.0)

create_typed_sql = """
DROP TABLE IF EXISTS trading_data_en;
CREATE TABLE trading_data_en (
    source_file VARCHAR,
    stock_code VARCHAR,
    stock_name VARCHAR,
    industry VARCHAR,
    region VARCHAR,
    ts_code VARCHAR,
    list_date DATE,
    trade_date DATE,
    open DOUBLE,
    high DOUBLE,
    low DOUBLE,
    close DOUBLE,
    prev_close DOUBLE,
    change_amt DOUBLE,
    change_pct DOUBLE,
    volume_lot DOUBLE,
    turnover_amount DOUBLE,
    turnover_pct DOUBLE,
    pe DOUBLE,
    pb DOUBLE,
    ps DOUBLE,
    dividend_yield_pct DOUBLE,
    free_float_mkt_cap DOUBLE,
    total_mkt_cap DOUBLE
);
"""
conn.execute(create_typed_sql)

insert_typed_sql = f"""
INSERT INTO trading_data_en
SELECT
    source_file::VARCHAR AS source_file,
    {text_expr(['股票代码'], 'stock_code')},
    {text_expr(['名称'], 'stock_name')},
    {text_expr(['所属行业'], 'industry')},
    {text_expr(['地域'], 'region')},
    {text_expr(['TS代码'], 'ts_code')},
    {date_expr(['上市日期'], 'list_date')},
    {date_expr(['交易日期'], 'trade_date')},
    {numeric_expr(['开盘价'], 'open')},
    {numeric_expr(['最高价'], 'high')},
    {numeric_expr(['最低价'], 'low')},
    {numeric_expr(['收盘价'], 'close')},
    {numeric_expr(['前收盘价'], 'prev_close')},
    {numeric_expr(['涨跌额'], 'change_amt')},
    {numeric_expr(['涨跌幅(%)'], 'change_pct')},
    {numeric_expr(['成交量(手)'], 'volume_lot')},
    {turnover_amount_expr},
    {numeric_expr(['换手率(%)'], 'turnover_pct')},
    {numeric_expr(['市盈率', '市盈率(TTM,亏损的PE为空)'], 'pe')},
    {numeric_expr(['市净率'], 'pb')},
    {numeric_expr(['市销率', '市销率(TTM)'], 'ps')},
    {numeric_expr(['股息率(%)'], 'dividend_yield_pct')},
    {free_float_expr},
    {total_mkt_cap_expr}
FROM trading_data_raw;
"""
conn.execute(insert_typed_sql)

conn.execute("CREATE OR REPLACE VIEW trading_data_clean AS SELECT * FROM trading_data_en")
print('Created table: trading_data_en (English columns with explicit schema)')
print('Created view: trading_data_clean -> trading_data_en')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created table: trading_data_en (English columns with explicit schema)
Created view: trading_data_clean -> trading_data_en


In [7]:
summary_df = conn.execute("""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT stock_code) AS stock_count,
    MIN(trade_date) AS min_trade_date,
    MAX(trade_date) AS max_trade_date
FROM trading_data_en
""").fetchdf()
display(summary_df)

null_check_df = conn.execute("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN trade_date IS NULL THEN 1 ELSE 0 END) AS null_trade_date_rows,
    SUM(CASE WHEN close IS NULL THEN 1 ELSE 0 END) AS null_close_rows,
    SUM(CASE WHEN total_mkt_cap IS NULL THEN 1 ELSE 0 END) AS null_total_mkt_cap_rows
FROM trading_data_en
""").fetchdf()
display(null_check_df)

schema_df = conn.execute("""
SELECT
    table_name,
    column_name,
    data_type
FROM information_schema.columns
WHERE table_name = 'trading_data_en'
ORDER BY ordinal_position
""").fetchdf()
display(schema_df)

sample_df = conn.execute("""
SELECT *
FROM trading_data_en
ORDER BY trade_date DESC NULLS LAST
LIMIT 20
""").fetchdf()
display(sample_df)

,row_count,stock_count,min_trade_date,max_trade_date
0,15800649,5739,1993-01-07,2025-12-05


,total_rows,null_trade_date_rows,null_close_rows,null_total_mkt_cap_rows
0,15800649,422097.0,11326.0,396011.0


,table_name,column_name,data_type
0,trading_data_en,source_file,VARCHAR
1,trading_data_en,stock_code,VARCHAR
2,trading_data_en,stock_name,VARCHAR
3,trading_data_en,industry,VARCHAR
4,trading_data_en,region,VARCHAR
5,trading_data_en,ts_code,VARCHAR
6,trading_data_en,list_date,DATE
7,trading_data_en,trade_date,DATE
8,trading_data_en,open,DOUBLE
9,trading_data_en,high,DOUBLE


,source_file,stock_code,stock_name,industry,region,ts_code,list_date,trade_date,open,high,...,change_pct,volume_lot,turnover_amount,turnover_pct,pe,pb,ps,dividend_yield_pct,free_float_mkt_cap,total_mkt_cap
0,000001.csv,1,平安银行,银行,深圳,000001.SZ,1991-04-03,2025-12-05,11.49,11.54,...,0.3481,890766.09,1.022335e+09,0.4590,5.0272,0.5046,1.5253,8.3695,2.237466e+11,2.237502e+11
1,000002.csv,2,万科A,全国地产,深圳,000002.SZ,1991-01-29,2025-12-05,4.94,4.97,...,-0.4024,3180996.40,1.552890e+09,3.2738,NaN,0.3360,0.1721,0.0000,4.809618e+10,5.905701e+10
2,000004.csv,4,国华网安,软件服务,深圳,000004.SZ,1991-01-14,2025-12-05,11.38,11.60,...,1.5762,27668.00,3.158796e+07,2.1909,NaN,41.5617,15.5646,0.0000,1.464938e+09,1.535611e+09
3,000006.csv,6,深振业A,区域地产,深圳,000006.SZ,1992-04-27,2025-12-05,9.84,10.28,...,2.2290,320332.57,3.209135e+08,2.3729,NaN,2.5240,2.2459,0.0000,1.362137e+10,1.362145e+10
4,000007.csv,7,全新好,其他商业,深圳,000007.SZ,1992-04-13,2025-12-05,10.15,10.15,...,-1.9704,90357.44,9.013533e+07,2.6081,61.0964,18.8707,11.2184,0.0000,3.447158e+09,3.447158e+09
5,000008.csv,8,神州高铁,运输设备,北京,000008.SZ,1992-05-07,2025-12-05,3.01,3.06,...,1.6667,503819.91,1.523328e+08,1.8548,NaN,2.8265,3.9793,0.0000,8.284706e+09,8.284952e+09
6,000009.csv,9,中国宝安,电气设备,深圳,000009.SZ,1991-06-25,2025-12-05,9.99,10.30,...,2.5000,315914.51,3.208604e+08,1.2264,153.1465,2.6232,1.3070,0.4390,2.640421e+10,2.643694e+10
7,000010.csv,10,美丽生态,建筑工程,深圳,000010.SZ,1995-10-27,2025-12-05,3.72,3.75,...,-0.2674,150381.00,5.597110e+07,1.8110,243.3526,13.1347,5.0918,0.0000,3.097361e+09,4.288203e+09
8,000011.csv,11,深物业A,房产服务,深圳,000011.SZ,1992-03-30,2025-12-05,9.35,9.44,...,0.8556,42788.00,4.001964e+07,0.8127,NaN,1.6560,2.0555,3.3086,4.964664e+09,5.620083e+09
9,000012.csv,12,南玻A,玻璃,深圳,000012.SZ,1992-02-28,2025-12-05,4.72,4.73,...,0.4246,128361.78,6.050052e+07,0.6551,54.4448,1.0962,0.9398,5.2854,9.267568e+09,1.452437e+10


In [8]:
conn.close()
print('DuckDB connection closed.')

DuckDB connection closed.
